# Index Directional — QuantDSL Strategy

Implements **TKAN v3 + IVol regime filter** using the QuantDSL declarative framework.

### Architecture
| Layer | Class | Role |
|---|---|---|
| Factors | `ExternalFactor` | TKAN v3 predictions (pkl cache, summed r1…r5) |
|         | `FieldFactor`     | CAC implied volatility (3m 50Δ) from sfera_db |
| Signals | `ZScoreRolling`  | IVol z-score over 126-day window |
|         | `MaskFromBoolean` | Entry signal (TKAN pos AND IVol z < 1.0) |
| Portfolio | `TimingPortfolio` | Long/flat CAC-TR (1× via CACT) |

### DSL Extension notes
The following nodes were **added to the DSL** to support this strategy:
- `ExternalFactor` — load pre-computed ML outputs from pickle/parquet/CSV  
- `FieldFactor` — expose auxiliary data column as a factor  
- `TimingPortfolio` — single-instrument long/flat timing (replaces `LongShortPortfolio`)  
- `MaskSelector` — select instruments matching a boolean signal  

The `TimingRunner` class in Cell 4 is a lightweight DSL interpreter that executes a
`Strategy(portfolio=TimingPortfolio)` directly on pandas — no need for the full
`BacktestRunner` event loop used by cross-sectional strategies.

In [ ]:
# ── 0. Setup ─────────────────────────────────────────────────────────────────

import os, sys, pathlib, pickle, json, hashlib, warnings
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import numpy as np
import pandas as pd

# ── Paths ─────────────────────────────────────────────────────────────────────
_HERE       = pathlib.Path(".").resolve()
_BTEST_ROOT = _HERE.parents[1]          # btest/
_WS_ROOT    = _BTEST_ROOT.parent        # workspace root
_TKAN_V3    = _HERE / "tkan" / "v3"
_WEIGHTS    = _TKAN_V3 / "weights"
_OUTPUT     = _BTEST_ROOT / "outputs" / "idx_directional"
_OUTPUT.mkdir(parents=True, exist_ok=True)

# Add workspace root so sfera_db and signum are importable
for p in [str(_WS_ROOT), str(_WS_ROOT / "sfera-db"), str(_WS_ROOT / "signum")]:
    if p not in sys.path:
        sys.path.insert(0, p)

# ── QuantDSL imports ─────────────────────────────────────────────────────────
from quantdsl_backtest.dsl.strategy     import Strategy
from quantdsl_backtest.dsl.data_config  import DataConfig
from quantdsl_backtest.dsl.universe     import Universe
from quantdsl_backtest.dsl.factors      import ExternalFactor, FieldFactor
from quantdsl_backtest.dsl.signals      import (
    ZScoreRolling, MaskFromBoolean,
    GreaterEqual, LessEqual, Less, Greater, And, Or, Not,
    RollingMean, RollingStd, EWMMean,
)
from quantdsl_backtest.dsl.portfolio    import TimingPortfolio
from quantdsl_backtest.dsl.execution    import Execution, OrderPolicy, LatencyModel, PowerLawSlippageModel, VolumeParticipation
from quantdsl_backtest.dsl.costs        import Costs, Commission, BorrowCost, FinancingCost, StaticFees
from quantdsl_backtest.dsl.backtest_config import BacktestConfig, Reporting

# ── Strategy constants ────────────────────────────────────────────────────────
BACKTEST_START  = "2015-01-01"
IVOL_WINDOW     = 126     # rolling z-score window for IVol regime filter
IVOL_Z_THRESH   = 1.0     # z-score threshold: < IVOL_Z_THRESH → favourable regime
TKAN_THRESH     = 0.0     # cumulative 5d prediction ≥ TKAN_THRESH → bullish
WINDOW_SIZE     = 30      # TKAN input sequence length (must match training)
PREDICTION_DAYS = 5       # TKAN output horizon (must match training)
FEATURE_COLS    = [
    "log_return_1d", "high_low_range", "close_to_high",
    "close_vs_sma15",
    "ivol_zscore", "ivol_ema_ratio", "ivol_pctl", "ivol_roc5",
    "rvol_park20_zscore", "vol_spread",
    "return_5d", "return_20d",
]

print("✅  Imports OK —", _BTEST_ROOT.name)

In [ ]:
# ── 1. DSL Strategy Definition ─────────────────────────────────────────────
#
# This cell is PURELY DECLARATIVE.  No data is loaded here.
# All node names must be unique keys used in the factors/signals dicts.
# The TimingRunner in Cell 4 will interpret these objects.

# --- 1a. Data source (documents intent; TimingRunner uses its own loader)
data = DataConfig(
    source="sfera://bbgidx/index_prices",
    calendar="XPAR",
    frequency="1d",
    start=BACKTEST_START,
    end="2025-12-31",
    fields=["open", "high", "low", "close", "volume", "3m_50d_ivol"],
)

# --- 1b. Universe: single instrument (hard-coded)
universe = Universe(name="CAC_TR", static_instruments=["CACT"])

# --- 1c. Factors
tkan_pred = ExternalFactor(
    name="tkan_pred",
    path=str(_WEIGHTS / "pred_cache.pkl"),  # (pred_df, retrain_dates, fingerprint)
    column=None,   # DataFrame with r1..r5; runner sums them to get cumulative view
)
ivol_raw = FieldFactor(
    name="ivol_raw",
    field="ivol",   # column name as loaded by load_data() below
)

# --- 1d. Signals
# IVol regime filter: rolling 126d z-score < 1.0  (low-vol environment favourable)
ivol_z = ZScoreRolling(
    name="ivol_z",
    base="ivol_raw",
    window=IVOL_WINDOW,
    min_periods=IVOL_WINDOW // 2,
)
ivol_ok = MaskFromBoolean(
    name="ivol_ok",
    expr=Less(left="ivol_z", right=IVOL_Z_THRESH),
)

# TKAN signal: cumulative 5d predicted return ≥ threshold  (net bullish)
tkan_ok = MaskFromBoolean(
    name="tkan_ok",
    expr=GreaterEqual(left="tkan_pred", right=TKAN_THRESH),
)

# Combined entry: both conditions hold
entry_signal = MaskFromBoolean(
    name="entry_signal",
    expr=And(left="tkan_ok", right="ivol_ok"),
)

# --- 1e. Portfolio: timing, single instrument
portfolio = TimingPortfolio(
    signal_name="entry_signal",
    instrument="CACT",
    rebalance_frequency="1d",
    rebalance_at="market_close",
    signal_delay_bars=1,    # signal at close[T] → position from close[T] to close[T+1]
    target_leverage=1.0,
)

# --- 1f. Execution & costs (for documentation; timing runner applies simple cost model)
execution = Execution(
    order_policy=OrderPolicy(),
    latency=LatencyModel(),
    slippage=PowerLawSlippageModel(base_bps=2.0, k=0.0),
    volume_limits=VolumeParticipation(max_participation=1.0),
)
costs = Costs(
    commission=Commission(type="bps_notional", amount=2.0),
    borrow=BorrowCost(),
    financing=FinancingCost(),
    fees=StaticFees(),
)
bt = BacktestConfig(
    reporting=Reporting(output_dir=str(_OUTPUT)),
)

# --- 1g. Compose
strategy = Strategy(
    name="index_directional",
    data=data,
    universe=universe,
    factors={
        "tkan_pred": tkan_pred,
        "ivol_raw":  ivol_raw,
    },
    signals={
        "ivol_z":       ivol_z,
        "ivol_ok":      ivol_ok,
        "tkan_ok":      tkan_ok,
        "entry_signal": entry_signal,
    },
    portfolio=portfolio,
    execution=execution,
    costs=costs,
    backtest=bt,
)

print("✅  Strategy object built:")
print(f"    name         : {strategy.name}")
print(f"    instrument   : {strategy.portfolio.instrument}")
print(f"    signal       : {strategy.portfolio.signal_name}")
print(f"    delay bars   : {strategy.portfolio.signal_delay_bars}")
print(f"    factors      : {list(strategy.factors.keys())}")
print(f"    signals      : {list(strategy.signals.keys())}")
print(f"    portfolio    : {type(strategy.portfolio).__name__}")

In [ ]:
# ── 2. TimingRunner — lightweight DSL interpreter for TimingPortfolio ────────
#
# Interprets a Strategy(portfolio=TimingPortfolio) directly on pandas.
# Does NOT use the full QuantDSL BacktestRunner event loop.
# Supports: ExternalFactor, FieldFactor, ZScoreRolling, MaskFromBoolean,
#           GreaterEqual, LessEqual, Less, Greater, And, Or, Not,
#           RollingMean, RollingStd, EWMMean

class TimingRunner:
    """
    Interpret a Strategy(portfolio=TimingPortfolio) on pandas.

    Usage::

        runner = TimingRunner(strategy, _BTEST_ROOT)
        result = runner.run(
            price_close = df["close"],   # pd.Series, DatetimeIndex
            aux_series  = {"ivol": df["ivol"]},   # dict[str, pd.Series]
        )
        # result keys: position, entry, price, daily_ret, strat_ret, factors, signals
    """

    def __init__(self, strategy: Strategy, btest_root):
        assert isinstance(strategy.portfolio, TimingPortfolio), (
            f"TimingRunner only handles TimingPortfolio, got {type(strategy.portfolio).__name__}"
        )
        self.strategy   = strategy
        self.btest_root = pathlib.Path(btest_root)

    # ------------------------------------------------------------------
    # Public API

    def run(self, price_close: pd.Series, aux_series: dict | None = None) -> dict:
        """
        Parameters
        ----------
        price_close:  pd.Series  — close prices for the instrument, DatetimeIndex
        aux_series:   dict[str, pd.Series]  — extra columns needed by FieldFactor
        """
        port = self.strategy.portfolio
        idx  = price_close.index

        # 1. Resolve factors
        computed: dict[str, pd.Series] = {}
        for name, fnode in self.strategy.factors.items():
            computed[name] = self._eval_factor(fnode, idx, aux_series or {})

        # 2. Evaluate signals
        signal_vals: dict[str, pd.Series] = {}
        for name, snode in self.strategy.signals.items():
            signal_vals[name] = self._eval_signal(snode, computed, signal_vals)

        # 3. Entry mask → position (with delay)
        entry    = signal_vals[port.signal_name].reindex(idx).fillna(False).astype(bool)
        position = entry.shift(port.signal_delay_bars).fillna(False).astype(int)

        # 4. Returns
        daily_ret = np.log(price_close / price_close.shift(1))
        strat_ret = (position * daily_ret).fillna(0.0)

        return dict(
            position  = position,
            entry     = entry,
            price     = price_close,
            daily_ret = daily_ret.fillna(0.0),
            strat_ret = strat_ret,
            factors   = computed,
            signals   = signal_vals,
        )

    # ------------------------------------------------------------------
    # Factor evaluation

    def _eval_factor(self, fnode, idx, aux_series: dict) -> pd.Series:
        if isinstance(fnode, ExternalFactor):
            return self._load_external(fnode, idx)
        if isinstance(fnode, FieldFactor):
            if fnode.field not in aux_series:
                raise ValueError(
                    f"FieldFactor '{fnode.name}' requires field '{fnode.field}' "
                    f"in aux_series.  Provided keys: {list(aux_series.keys())}"
                )
            return aux_series[fnode.field].reindex(idx)
        raise NotImplementedError(f"_eval_factor: unsupported {type(fnode).__name__}")

    def _load_external(self, fnode, idx) -> pd.Series:
        """Load ExternalFactor from disk.  Handles TKAN pred_cache tuple format."""
        path = pathlib.Path(fnode.path)
        if not path.is_absolute():
            path = self.btest_root / path
        if not path.exists():
            raise FileNotFoundError(f"ExternalFactor '{fnode.name}': file not found: {path}")

        with open(path, "rb") as f:
            obj = pickle.load(f)

        # TKAN pred_cache format: (pred_df_r1r5, retrain_dates, fingerprint)
        if isinstance(obj, tuple) and len(obj) >= 1 and isinstance(obj[0], pd.DataFrame):
            pred_df = obj[0]          # columns r1..r5
            if fnode.column and fnode.column in pred_df.columns:
                series = pred_df[fnode.column]
            else:
                series = pred_df.sum(axis=1)  # cumulative 5d prediction
        elif isinstance(obj, pd.DataFrame):
            col    = fnode.column or obj.columns[0]
            series = obj[col]
        elif isinstance(obj, pd.Series):
            series = obj
        else:
            raise TypeError(
                f"ExternalFactor '{fnode.name}': unsupported pickle type {type(obj).__name__}"
            )

        series.index = pd.DatetimeIndex(series.index)
        return series.reindex(idx)

    # ------------------------------------------------------------------
    # Signal / expression evaluation

    def _resolve(self, expr, computed, signal_vals) -> pd.Series | float:
        """Resolve an Expr reference to a concrete Series or scalar."""
        if isinstance(expr, str):
            if expr in signal_vals:
                return signal_vals[expr]
            if expr in computed:
                return computed[expr]
            raise KeyError(f"Expression '{expr}' not found in factors or signals")
        if isinstance(expr, (int, float)):
            return expr
        # Nested SignalNode
        return self._eval_signal(expr, computed, signal_vals)

    def _eval_signal(self, snode, computed, signal_vals) -> pd.Series:
        """Evaluate a SignalNode, returning a pd.Series."""
        r = self._resolve

        if isinstance(snode, ZScoreRolling):
            base = r(snode.base, computed, signal_vals)
            win  = snode.window
            mp   = getattr(snode, "min_periods", 1)
            mu   = base.rolling(win, min_periods=mp).mean()
            sd   = base.rolling(win, min_periods=mp).std()
            return (base - mu) / (sd + 1e-9)

        if isinstance(snode, MaskFromBoolean):
            return r(snode.expr, computed, signal_vals).astype(bool)

        if isinstance(snode, GreaterEqual):
            return r(snode.left, computed, signal_vals) >= r(snode.right, computed, signal_vals)

        if isinstance(snode, LessEqual):
            return r(snode.left, computed, signal_vals) <= r(snode.right, computed, signal_vals)

        if isinstance(snode, Less):
            return r(snode.left, computed, signal_vals) < r(snode.right, computed, signal_vals)

        if isinstance(snode, Greater):
            return r(snode.left, computed, signal_vals) > r(snode.right, computed, signal_vals)

        if isinstance(snode, And):
            left  = r(snode.left,  computed, signal_vals)
            right = r(snode.right, computed, signal_vals)
            return left.astype(bool) & right.astype(bool)

        if isinstance(snode, Or):
            left  = r(snode.left,  computed, signal_vals)
            right = r(snode.right, computed, signal_vals)
            return left.astype(bool) | right.astype(bool)

        if isinstance(snode, Not):
            return ~r(snode.expr, computed, signal_vals).astype(bool)

        if isinstance(snode, RollingMean):
            base = r(snode.base, computed, signal_vals)
            return base.rolling(snode.window, min_periods=getattr(snode, "min_periods", 1)).mean()

        if isinstance(snode, RollingStd):
            base = r(snode.base, computed, signal_vals)
            return base.rolling(snode.window, min_periods=getattr(snode, "min_periods", 1)).std()

        if isinstance(snode, EWMMean):
            base = r(snode.base, computed, signal_vals)
            return base.ewm(
                span=snode.span,
                min_periods=getattr(snode, "min_periods", 1),
                adjust=getattr(snode, "adjust", False),
            ).mean()

        raise NotImplementedError(
            f"TimingRunner._eval_signal: unsupported node '{type(snode).__name__}'. "
            f"Add it to the elif chain to extend support."
        )


runner = TimingRunner(strategy, _BTEST_ROOT)
print("✅  TimingRunner ready")

In [ ]:
# ── 3. Load data from sfera_db ────────────────────────────────────────────────

import sfera_db

cactr = sfera_db.query(
    "SELECT trade_date AS date, close_price AS close "
    "FROM bbgidx.index_total_return WHERE ticker = 'CACT' ORDER BY trade_date"
).assign(date=lambda d: pd.to_datetime(d["date"])).set_index("date")

cac_ohlc = sfera_db.query(
    "SELECT trade_date AS date, open_price AS open, high_price AS high, "
    "low_price AS low, close_price AS cac_close "
    "FROM bbgidx.index_prices WHERE ticker = 'CAC' ORDER BY trade_date"
).assign(date=lambda d: pd.to_datetime(d["date"])).set_index("date")

ivol_raw_db = sfera_db.query(
    'SELECT trade_date AS date, "3m_50d_ivol" AS ivol '
    "FROM bbgidx.index_implied_vol WHERE ticker = 'CAC' ORDER BY trade_date"
).assign(date=lambda d: pd.to_datetime(d["date"])).set_index("date")[["ivol"]]

# Align on common dates
common = cactr.index.intersection(cac_ohlc.index).intersection(ivol_raw_db.index)
df = cac_ohlc.loc[common].copy()
df["close"] = cactr.loc[common, "close"]
df["ivol"]  = ivol_raw_db.loc[common, "ivol"]

# ── Feature engineering (match TKAN training notebook exactly) ───────────────
df["log_return_1d"]  = np.log(df["close"] / df["close"].shift(1))
df["high_low_range"] = (df["high"] - df["low"]) / df["close"].shift(1).replace(0, np.nan)
df["close_to_high"]  = (df["high"] - df["cac_close"]) / (df["high"] - df["low"] + 1e-9)
df["sma15"]          = df["close"].rolling(15).mean()
df["close_vs_sma15"] = df["close"] / df["sma15"] - 1
df["return_5d"]      = np.log(df["close"] / df["close"].shift(5))
df["return_20d"]     = np.log(df["close"] / df["close"].shift(20))

hl_sq               = np.log(df["high"] / df["low"].replace(0, np.nan)) ** 2
park                = np.sqrt((1 / (4 * np.log(2))) * hl_sq.rolling(20).mean() * 252)
close_rvol          = df["log_return_1d"].rolling(20).std() * np.sqrt(252)
df["rvol_park20"]   = park.where(park.notna() & (park > 0), close_rvol)

ivol = df["ivol"]
df["ivol_ewma20"]        = ivol.ewm(span=20).mean()
df["ivol_zscore"]        = (ivol - ivol.rolling(IVOL_WINDOW).mean()) / (ivol.rolling(IVOL_WINDOW).std() + 1e-9)
df["ivol_ema_ratio"]     = ivol / (df["ivol_ewma20"] + 1e-9)
df["ivol_pctl"]          = ivol.rolling(IVOL_WINDOW).apply(lambda x: pd.Series(x).rank(pct=True).iloc[-1], raw=False)
df["ivol_roc5"]          = ivol.pct_change(5)
df["rvol_park20_zscore"] = (df["rvol_park20"] - df["rvol_park20"].rolling(IVOL_WINDOW).mean()) / \
                            (df["rvol_park20"].rolling(IVOL_WINDOW).std() + 1e-9)
df["vol_spread"]         = ivol - df["rvol_park20"]

# Trim to backtest window
df = df.loc[df.index >= pd.Timestamp(BACKTEST_START)].copy()

print(f"✅  Data loaded: {len(df):,} rows  "
      f"{df.index[0].date()} → {df.index[-1].date()}")
print(f"    columns: {list(df.columns)}")

In [ ]:
# ── 4. Run DSL strategy via TimingRunner ──────────────────────────────────────

result = runner.run(
    price_close = df["close"],
    aux_series  = {"ivol": df["ivol"]},
)

position    = result["position"]
entry       = result["entry"]
strat_ret   = result["strat_ret"]
daily_ret   = result["daily_ret"]
tkan_score  = result["factors"]["tkan_pred"]
ivol_z_vals = result["signals"]["ivol_z"]

# Equity curves
equity_bh     = (1 + daily_ret).cumprod()
equity_strat  = (1 + strat_ret).cumprod()

in_market_pct = 100 * position.mean()
print(f"✅  Strategy run complete")
print(f"    in-market  : {in_market_pct:.1f}%")
print(f"    data range : {strat_ret.index[0].date()} → {strat_ret.index[-1].date()}")
print(f"    tkan_pred  : {tkan_score.notna().sum():,} non-null rows")

In [ ]:
# ── 5. Metrics — compare 4 signal variants ───────────────────────────────────
#
# Variants:
#   B&H       — buy and hold (full period)
#   IVol z    — long when ivol_z < IVOL_Z_THRESH
#   TKAN      — long when tkan_pred >= TKAN_THRESH
#   TKAN+IVol — combined (DSL entry_signal)

from scipy import stats as scipy_stats

def compute_metrics(strat_ret: pd.Series, bh_ret: pd.Series, pos: pd.Series, label: str) -> dict:
    sr = strat_ret.fillna(0)
    br = bh_ret.reindex(sr.index).fillna(0)
    n_years = len(sr) / 252

    ann_ret = sr.mean() * 252
    ann_vol = sr.std()  * np.sqrt(252)
    sharpe  = ann_ret / ann_vol if ann_vol > 0 else 0.0
    down    = sr[sr < 0]
    sortino = ann_ret / (down.std() * np.sqrt(252)) if len(down) > 1 else 0.0

    equity = (1 + sr).cumprod()
    mdd    = float((equity / equity.cummax() - 1).min()) * 100
    total  = float(equity.iloc[-1]) if len(equity) else 1.0
    cagr   = (total ** (1 / n_years) - 1) * 100 if n_years > 0 else 0.0
    calmar = cagr / abs(mdd) if mdd != 0 else float("nan")

    valid  = br.notna() & sr.notna()
    if valid.sum() > 30:
        slope, intercept, *_ = scipy_stats.linregress(br[valid], sr[valid])
        beta  = float(slope)
        alpha = float(intercept) * 252 * 100
    else:
        beta = alpha = float("nan")

    active = sr[pos.reindex(sr.index).fillna(0) > 0]
    win_rate   = float((active > 0).mean() * 100) if len(active) > 0 else float("nan")
    in_mkt_pct = float(pos.reindex(sr.index).fillna(0).mean() * 100)

    return dict(
        Label=label, CAGR=round(cagr, 2), Sharpe=round(sharpe, 3),
        Sortino=round(sortino, 3), MaxDD=round(mdd, 2), Calmar=round(calmar, 3),
        Beta=round(beta, 3), Alpha=round(alpha, 2),
        WinPct=round(win_rate, 1), InMktPct=round(in_mkt_pct, 1),
        TotalReturn=round((total - 1) * 100, 1),
    )

bh_ret = daily_ret
bh_pos = pd.Series(1, index=bh_ret.index)

# IVol-only signal
ivol_sig  = (ivol_z_vals < IVOL_Z_THRESH).shift(1).fillna(False).astype(int)
ivol_ret  = (ivol_sig * daily_ret).fillna(0)

# TKAN-only signal
tkan_sig  = result["signals"]["tkan_ok"].shift(1).fillna(False).astype(int)
tkan_ret  = (tkan_sig * daily_ret).fillna(0)

rows = [
    compute_metrics(bh_ret,    bh_ret, bh_pos,   "Buy & Hold"),
    compute_metrics(ivol_ret,  bh_ret, ivol_sig,  "IVol z-score"),
    compute_metrics(tkan_ret,  bh_ret, tkan_sig,  "TKAN v3"),
    compute_metrics(strat_ret, bh_ret, position,  "TKAN + IVol (DSL)"),
]

metrics_df = pd.DataFrame(rows).set_index("Label")
print("\n" + "=" * 70)
print("  Index Directional — Signal Comparison")
print("=" * 70)
print(metrics_df.to_string())
print("=" * 70)

out_path = _OUTPUT / "dsl_metrics.csv"
metrics_df.to_csv(out_path)
print(f"\n saved → {out_path}")

In [ ]:
# ── 6. Signum — 3-pane interactive chart ─────────────────────────────────────
#
# Pane 1: CACT close price + green/grey shading (in/out of market)
# Pane 2: TKAN cumulative 5d prediction + IVol z-score
# Pane 3: Equity curves (B&H, IVol z, TKAN, TKAN+IVol)

try:
    from signum import Chart, Dashboard
except ImportError:
    print("⚠   signum not found — skipping chart")
    raise SystemExit(0)

# ── Prepare series ────────────────────────────────────────────────────────────
price       = df["close"]
eq_bh       = (1 + bh_ret).cumprod()
eq_ivol     = (1 + ivol_ret).cumprod()
eq_tkan     = (1 + tkan_ret).cumprod()
eq_combined = equity_strat

# ── Pane 1: Price ─────────────────────────────────────────────────────────────
pane1 = Chart(title="CACT — Close Price", theme="dark", height=280)
pane1.line(price.rename("CACT"), color="#7cb9e8", linewidth=1.5)
# Shade in-market periods
pane1.area(
    price.where(position.astype(bool)).rename("Long"),
    top_color="rgba(80,200,120,0.18)",
    bottom_color="rgba(80,200,120,0.04)",
)

# ── Pane 2: Signals ───────────────────────────────────────────────────────────
pane2 = Chart(title="TKAN pred (5d cum) | IVol z-score", theme="dark", height=220)
pane2.line(tkan_score.rename("TKAN pred"),  color="#f9a825", linewidth=1.0)
pane2.line(ivol_z_vals.rename("IVol z"),    color="#ef5350", linewidth=1.0)
# Zero / threshold reference lines
pane2.line(
    pd.Series(0.0, index=price.index, name="zero"),
    color="rgba(255,255,255,0.15)", linewidth=1, linestyle="dashed",
)
pane2.line(
    pd.Series(IVOL_Z_THRESH, index=price.index, name=f"IVol thr={IVOL_Z_THRESH}"),
    color="rgba(239,83,80,0.4)", linewidth=1, linestyle="dashed",
)

# ── Pane 3: Equity curves ─────────────────────────────────────────────────────
pane3 = Chart(title="Equity Curves (rebased to 1.0)", theme="dark", height=280)
pane3.line(eq_bh.rename("B&H"),            color="#78909c", linewidth=1.3)
pane3.line(eq_ivol.rename("IVol z"),       color="#ef5350", linewidth=1.3)
pane3.line(eq_tkan.rename("TKAN v3"),      color="#f9a825", linewidth=1.3)
pane3.line(eq_combined.rename("TKAN+IVol"), color="#66bb6a", linewidth=2.0)

# ── Render dashboard ──────────────────────────────────────────────────────────
dash = Dashboard(panes=[pane1, pane2, pane3], theme="dark")
dash.show()

In [ ]:
# ── 7. Threshold sweep — TKAN cutoff ─────────────────────────────────────────
#
# Sweep TKAN_THRESH from p5 to p75 of tkan_score distribution,
# with and without the IVol z-score gate

t_range = np.percentile(tkan_score.dropna(), np.arange(5, 76, 5))
rows = []

for thr in t_range:
    for ivol_gate in [False, True]:
        tkan_mask = (tkan_score >= thr)
        if ivol_gate:
            mask = tkan_mask & (ivol_z_vals < IVOL_Z_THRESH)
            label = f"TKAN+IVol  thr={thr:.4f}"
        else:
            mask = tkan_mask
            label = f"TKAN only  thr={thr:.4f}"
        pos_s  = mask.shift(1).fillna(False).astype(int)
        ret_s  = (pos_s * daily_ret).fillna(0)
        m      = compute_metrics(ret_s, bh_ret, pos_s, label)
        m["thr"]      = round(float(thr), 5)
        m["ivol_gate"] = ivol_gate
        rows.append(m)

sweep_df = pd.DataFrame(rows).set_index("Label")
print("\n── Threshold Sweep ──────────────────────────────────────────────────")
print(sweep_df[["thr", "ivol_gate", "Sharpe", "CAGR", "MaxDD", "Calmar", "InMktPct"]].to_string())